In [ ]:
#!pip install -U langchain-google-genai
#!pip install dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load = load_dotenv('../.env')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

tc_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
)

/Users/karthik/Downloads/agentsforQA_code-2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [3]:
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate
import os
from langchain_community.document_loaders import (
    UnstructuredPDFLoader
)


@tool
def generate_test_cases_from_pdf() -> str:
    "Reads the requirements from the PDF and generate atleast 10 test scenarios in BDD Format"
    
    docs_folder = "./Docs"
    documents = []

    for filename in os.listdir(docs_folder):
        filepath = os.path.join(docs_folder, filename)
        loader = UnstructuredPDFLoader(filepath)
        docs = loader.load();
        for doc in docs:
            documents.append(f"{filename}:\n{doc.page_content.strip()}")
    
    requirement_txt = "\n\n".join(documents[:3])[:1000]
    
   
    prompt_template = PromptTemplate.from_template(
        
        """
        You are are QA Automation Engineer.
        Your task is to convert the following user stories into at least 10 testcases, in Gherkin BDD style format.
        Include combinations of valid, invalid, edge case and alternative flow scearios
        
        {requirement_txt}
        
        Format:
        Feature: ...
        Scenario: ...
            Given ...
            When ...
            Then ...
        
        """
    )
    prompt = prompt_template.invoke({"requirement_txt":requirement_txt })
    return tc_llm.invoke(prompt)
    

In [ ]:
from langchain_classic.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools= [generate_test_cases_from_pdf],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

result = agent.invoke("Generate the test cases from PDF file by reading the requirement pdf documents")

print(result["output"])


/var/folders/jc/c7p4f2sd36xbqwkp2djn8flc0000gn/T/ipykernel_18541/3778711650.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(




> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "generate_test_cases_from_pdf",
  "action_input": {}
}
```

/Users/karthik/tryout/agentsforQA/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Observation: content='Okay, I\'ll create Gherkin BDD test cases based on the provided Netflix system design document. Since the document is a high-level overview, I\'ll focus on test cases that relate to the core functionalities that can be inferred from the document, such as transcoding, content delivery, and user authentication (which is implicitly required). I\'ll also create test cases based on a user trying to watch a movie.\n\n```gherkin\nFeature: Video Streaming on Netflix\n\n  Scenario: Successful video playback\n    Given the user is logged in\n    And the user has a valid subscription\n    And the movie "Example Movie" is available\n    When the user selects "Example Movie"\n    And the user clicks play\n    Then the video starts playing smoothly\n    And the playback controls are visible\n\n  Scenario: User attempts to play a movie without a subscription\n    Given the user is logged in\n    And the user does not have a valid subscription\n    And the movie "Example Movie" 